# Unidades Deliberativas Verificáveis: de citação exata a validação manual

Uma opinião do LDS é uma string dentro de uma lista, sem nenhum vínculo verificável com a transcrição de onde deveria ter vindo: não dá para responder quem disse aquilo, onde na fala, e com que confiança essa ligação pode ser feita. Este notebook percorre quatro tentativas de resolver esse problema, cada uma motivada pela limitação da anterior, e fecha com os defeitos do resolvedor de nomes expostos pela rodada completa nas 206 audiências, a correção deles e a política de casamento de citação que decide quando uma citação vale como evidência.

In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
with open("dataset/PublicHearingBR_LDS.jsonl") as f:
    lds_records = [json.loads(line) for line in f]

sample_hearings = lds_records[:20]

## Tentativa 1: evidência por citação direta

A forma mais simples de ligar uma opinião a uma evidência é procurar, dentro dela, um trecho entre aspas, e tentar localizar esse trecho na própria fala da pessoa. Antes disso, porém, é preciso conseguir separar a fala de cada participante dentro da transcrição.

In [3]:
import re

In [4]:
naive_turn_pattern = re.compile(
    r"(?:O\s+SR\.|A\s+SRA\.)\s*([A-ZÀ-Ü][^(\n]{0,60})\([^)]*\)\s*-\s"
)
sample_transcript = sample_hearings[0]["transcricao"]
naive_matches = naive_turn_pattern.findall(sample_transcript)
print(f"turnos encontrados exigindo parênteses: {len(naive_matches)}")
print("nomes únicos:", sorted({m.strip() for m in naive_matches}))

turnos encontrados exigindo parênteses: 53
nomes únicos: ['ADRIANA VENTURA', 'AFONSO HAMM', 'ARLINDO CHINAGLIA', 'BIA KICIS', 'FILIPE BARROS', 'FLORENTINO NETO', 'GENERAL GIRÃO', 'GILVAN DA FEDERAL', 'LUIZ PHILIPPE DE ORLEANS E BRAGANÇA', 'MARCEL VAN HATTEM', 'MARCELO MORAES', 'MARIO FRIAS', 'PRESIDENTE']


In [5]:
missing_person = "Michael Shellenberger"
position = sample_transcript.upper().find(missing_person.split()[0].upper(), 5000)
print(sample_transcript[position - 20 : position + 60])

laborador americano Michael Shellenberger.

Confesso que eu nem sei ao certo a o


O nome aparece, mas sem parênteses depois: convidados sem filiação partidária (especialistas, testemunhas estrangeiras) têm um cabeçalho de turno diferente. Um regex que exige parênteses parece funcionar, porque encontra turnos em toda audiência, mas exclui esses participantes silenciosamente. A pipeline compartilhada do projeto já corrige isso.

In [6]:
from utils.udv_pipeline import (
    best_semantic_match,
    extract_quotes,
    find_opinion_quote_evidence,
    quote_prefix_pattern,
    resolve_person_speech,
    split_into_turns,
    split_sentences,
)


In [7]:
fixed_turns = split_into_turns(sample_transcript)
print(f"turnos encontrados com o regex corrigido: {len(fixed_turns)}")


turnos encontrados com o regex corrigido: 65


Com a segmentação de turnos corrigida, o próximo passo é resolver, para cada participante listado na metadata, quais turnos são dele, e então tentar achar uma citação direta de cada opinião na fala correspondente.

In [8]:
people_total = 0
people_resolved = 0
quote_only_counts = {"no_quote": 0, "quote_not_found": 0, "quote_found": 0, "person_not_resolved": 0}

for hearing in sample_hearings:
    turns = split_into_turns(hearing["transcricao"])
    for person in hearing["metadados"]["envolvidos"]:
        people_total += 1
        matched_turns, speech = resolve_person_speech(person, turns)
        if matched_turns:
            people_resolved += 1
        for opinion_text in person["opinioes"]:
            if not matched_turns:
                quote_only_counts["person_not_resolved"] += 1
                continue
            if not extract_quotes(opinion_text):
                quote_only_counts["no_quote"] += 1
            elif find_opinion_quote_evidence(opinion_text, speech) is not None:
                quote_only_counts["quote_found"] += 1
            else:
                quote_only_counts["quote_not_found"] += 1

In [9]:
print(f"pessoas resolvidas: {people_resolved}/{people_total}")
pd.Series(quote_only_counts, name="opinioes")

pessoas resolvidas: 102/108


no_quote               152
quote_not_found         65
quote_found             38
person_not_resolved     13
Name: opinioes, dtype: int64

Resolver a pessoa por trás de cada turno funciona bem (102 de 108 participantes, com as regras de resolução da pipeline compartilhada, incluindo as correções descritas na seção final deste notebook). Mas ligar cada opinião a uma citação localizada funciona mal: das 255 opiniões de participantes resolvidos (as outras 13 são de pessoas não resolvidas), 152 nem têm uma citação direta entre aspas (são paráfrase da matéria), 65 têm citação mas ela não foi localizada na fala da pessoa, e 38 tiveram a citação efetivamente encontrada. Buscar por correspondência de texto cobre uma fração pequena do problema, porque a maioria das opiniões é reformulada, não copiada. O que conta como citação localizada aqui já segue a política da última seção: prefixo casado sem distinção de maiúsculas e com 6 palavras ou mais.

## Tentativa 2: similaridade textual (TF-IDF)

Se a maioria das opiniões é paráfrase, o próximo passo natural é comparar significado por vocabulário compartilhado, não por texto idêntico. TF-IDF é exatamente o baseline de comparação textual já previsto para o projeto, não uma escolha arbitrária. Para isso, a fala de cada pessoa é dividida em sentenças, e cada opinião é comparada contra essas sentenças.

In [10]:
sample_speech = resolve_person_speech(
    sample_hearings[0]["metadados"]["envolvidos"][0], fixed_turns
)[1]
sample_sentences = split_sentences(sample_speech)
print(f"sentenças extraídas do primeiro participante: {len(sample_sentences)}")

sentenças extraídas do primeiro participante: 72


A comparação de cada opinião contra as sentenças usa `TfidfVectorizer` e similaridade de cosseno, guardando a sentença de maior similaridade e o escore. Mas um escore de similaridade sozinho não diz o que é uma evidência confiável: para isso, é preciso calibrar um corte.

In [11]:
import random

random.seed(42)

In [12]:
positive_scores = []
negative_scores = []
all_sentences_pool = []

for hearing in sample_hearings:
    turns = split_into_turns(hearing["transcricao"])
    person_sentences = {}
    for person in hearing["metadados"]["envolvidos"]:
        _, speech = resolve_person_speech(person, turns)
        sentences = split_sentences(speech)
        person_sentences[person["nome"]] = sentences
        all_sentences_pool.extend(sentences)

    for person in hearing["metadados"]["envolvidos"]:
        sentences = person_sentences[person["nome"]]
        if not sentences:
            continue
        for opinion_text in person["opinioes"]:
            found = find_opinion_quote_evidence(opinion_text, " ".join(sentences))
            if found is None:
                continue
            true_sentence = next((s for s in sentences if quote_prefix_pattern(found).search(s)), None)
            if true_sentence is None:
                continue

            _, positive_score = best_semantic_match(opinion_text, [true_sentence])
            positive_scores.append(positive_score)

            other_sentence = random.choice([s for s in all_sentences_pool if s != true_sentence])
            _, negative_score = best_semantic_match(opinion_text, [other_sentence])
            negative_scores.append(negative_score)

In [13]:
positive_series = pd.Series(positive_scores, name="par_correto")
negative_series = pd.Series(negative_scores, name="par_aleatorio")
pd.DataFrame({"correto": positive_series.describe(), "aleatorio": negative_series.describe()})

,correto,aleatorio
count,35.000000,35.000000
mean,0.520651,0.055779
std,0.170496,0.063785
min,0.147847,0.000000
25%,0.408150,0.000000
50%,0.530687,0.043569
75%,0.612256,0.072879
max,0.834301,0.275939


In [14]:
confidence_threshold = 0.25
positives_below = (positive_series < confidence_threshold).sum()
negatives_above = (negative_series >= confidence_threshold).sum()
print(f"corte escolhido: {confidence_threshold}")
print(f"pares corretos abaixo do corte: {positives_below}/{len(positive_series)}")
print(f"pares aleatórios acima do corte: {negatives_above}/{len(negative_series)}")

corte escolhido: 0.25
pares corretos abaixo do corte: 2/35
pares aleatórios acima do corte: 1/35


A mediana dos pares corretos fica em torno de 0,53, a dos pares aleatórios abaixo de 0,05: as duas distribuições se separam bem, com alguma sobreposição na faixa baixa (2 dos 35 pares corretos ficam abaixo de 0,25; 1 dos 35 aleatórios fica acima, em 0,28). Um corte de 0,25 é uma escolha razoável, não perfeita. Com esse corte, a classificação de cada opinião passa a ter quatro níveis: citação encontrada, similaridade acima do corte, similaridade abaixo do corte (uma tentativa de baixa confiança) ou pessoa não resolvida.

In [15]:
def classify_evidence(opinion_text, person_speech, sentences):
    if find_opinion_quote_evidence(opinion_text, person_speech) is not None:
        return "quote_found"

    best_sentence, score = best_semantic_match(opinion_text, sentences)
    if best_sentence is None:
        return "no_evidence"
    return "semantic_match_high" if score >= confidence_threshold else "semantic_match_weak"

In [16]:
coverage_v1 = {"quote_found": 0, "semantic_match_high": 0, "semantic_match_weak": 0, "person_not_resolved": 0}

for hearing in sample_hearings:
    turns = split_into_turns(hearing["transcricao"])
    for person in hearing["metadados"]["envolvidos"]:
        matched_turns, speech = resolve_person_speech(person, turns)
        sentences = split_sentences(speech)
        for opinion_text in person["opinioes"]:
            if not matched_turns:
                coverage_v1["person_not_resolved"] += 1
                continue
            coverage_v1[classify_evidence(opinion_text, speech, sentences)] += 1

In [17]:
pd.DataFrame(
    {"so_citacao": pd.Series(quote_only_counts), "citacao_mais_similaridade": pd.Series(coverage_v1)}
).fillna(0)

,so_citacao,citacao_mais_similaridade
no_quote,152.0,0.0
person_not_resolved,13.0,13.0
quote_found,38.0,38.0
quote_not_found,65.0,0.0
semantic_match_high,0.0,153.0
semantic_match_weak,0.0,64.0


Das 268 opiniões, 191 (38 por citação, 153 por similaridade acima do corte) terminam com evidência de confiança razoável, e as outras 64 recebem ao menos uma tentativa de baixa confiança em vez de nada. É uma melhora real. Mas a calibração só mostra que pares já sabidamente certos têm similaridade maior do que pares aleatórios, ela não mede a taxa de acerto das evidências que a pipeline de fato produz no dia a dia, quando não há citação para conferir.

## Tentativa 3: a similaridade é confiável? Validação manual

Para medir isso de verdade, uma amostra de pares opinião-evidência dos dois níveis de confiança foi classificada manualmente num arquivo revisável (`udv_manual_review.json`), sujeito a correção: cada par recebeu `correta`, `parcial` (mesmo assunto, mas não confirma a afirmação específica) ou `incorreta`.

In [18]:
with open("udv_manual_review.json") as f:
    review_items = json.load(f)

review_df = pd.DataFrame(review_items)
review_df = review_df[review_df["tier"] != "embedding_diff"]
print(f"itens carregados: {len(review_df)}")

itens carregados: 40


In [19]:
pd.crosstab(review_df["tier"], review_df["judgment"])

judgment,correta,incorreta,parcial
tier,,,
semantic_match_high,17,0,3
semantic_match_weak,3,9,8


In [20]:
accuracy_by_tier = review_df.groupby("tier")["judgment"].apply(lambda j: (j == "correta").mean())
accuracy_by_tier

tier
semantic_match_high    0.85
semantic_match_weak    0.15
Name: judgment, dtype: float64

Numa amostra de 20 pares por nível: `semantic_match_high` teve 17 corretos e 3 parciais, nenhum incorreto (85% de acerto). `semantic_match_weak` teve só 3 corretos, 8 parciais e 9 incorretos (15% de acerto, quase metade claramente errada). O corte de 0,25 separa bem evidência confiável de não confiável. A classificação vem de um arquivo revisável, editável à mão, sem revisão por um especialista humano independente. A amostra tem 20 pares por nível: o resultado é um indício direcional, sem intervalo de confiança formal.

## Para onde isso aponta

TF-IDF compara vocabulário compartilhado, não significado. Isso explica a taxa de erro alta na camada de baixa confiança: uma opinião que parafraseia a fala trocando palavras por sinônimos, mantendo o mesmo sentido, tende a ter similaridade baixa mesmo quando o conteúdo está correto.

O próximo passo é testar uma comparação por significado, com embeddings densos, na mesma amostra de validação manual, para ver se ela recupera os casos que caem na camada fraca por causa dessa limitação de vocabulário.

## Tentativa 4: comparação por significado com embeddings densos

Um modelo multilíngue genérico não é a melhor opção disponível para português: o benchmark MTEB-PT mostra que rankings multilíngues não predizem bem o desempenho específico em português, e no benchmark ASSIN2 (similaridade textual em português) o BERTimbau-Large chega a 0,852 de correlação de Pearson contra 0,809 de um BERT multilíngue genérico. O artigo que descreve a família Serafim PT* distingue as duas variantes do idioma e reporta que o Serafim 335, baseado no BERTimbau (português brasileiro), lidera nos conjuntos brasileiros, enquanto o Serafim 900 (baseado no Albertina, português europeu) é melhor nos conjuntos europeus. Como as transcrições do PublicHearingBR são em português do Brasil, o modelo usado aqui é `PORTULAN/serafim-335m-portuguese-pt-sentence-encoder`.

In [21]:
from utils.udv_pipeline import best_embedding_match, get_embedding_model

In [22]:
embedding_model = get_embedding_model()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

O modelo carregou diretamente com `SentenceTransformer`, sem precisar de mean pooling manual.

A calibração segue o mesmo método já usado com TF-IDF: comparar a similaridade em pares já confirmados por citação contra pares aleatórios, agora usando embeddings em vez de TF-IDF.

In [23]:
embedding_positive_scores = []
embedding_negative_scores = []

for hearing in sample_hearings:
    turns = split_into_turns(hearing["transcricao"])
    person_sentences = {}
    for person in hearing["metadados"]["envolvidos"]:
        _, speech = resolve_person_speech(person, turns)
        person_sentences[person["nome"]] = split_sentences(speech)

    for person in hearing["metadados"]["envolvidos"]:
        sentences = person_sentences[person["nome"]]
        if not sentences:
            continue
        for opinion_text in person["opinioes"]:
            found = find_opinion_quote_evidence(opinion_text, " ".join(sentences))
            if found is None:
                continue
            true_sentence = next((s for s in sentences if quote_prefix_pattern(found).search(s)), None)
            if true_sentence is None:
                continue

            _, positive_score = best_embedding_match(opinion_text, [true_sentence], embedding_model)
            embedding_positive_scores.append(positive_score)

            other_sentence = random.choice([s for s in all_sentences_pool if s != true_sentence])
            _, negative_score = best_embedding_match(opinion_text, [other_sentence], embedding_model)
            embedding_negative_scores.append(negative_score)

In [24]:
embedding_positive_series = pd.Series(embedding_positive_scores, name="par_correto")
embedding_negative_series = pd.Series(embedding_negative_scores, name="par_aleatorio")
pd.DataFrame(
    {
        "correto": embedding_positive_series.describe(),
        "aleatorio": embedding_negative_series.describe(),
    }
)

,correto,aleatorio
count,35.000000,35.000000
mean,0.730807,0.210878
std,0.140615,0.086670
min,0.411320,-0.002078
25%,0.668068,0.152136
50%,0.771539,0.191871
75%,0.842378,0.265024
max,0.909923,0.389673


A escala da similaridade de embeddings não é a mesma do TF-IDF, então o corte precisa ser recalculado a partir desta distribuição, não reaproveitado do corte de 0,25.

In [25]:
embedding_threshold = round(
    (embedding_positive_series.quantile(0.25) + embedding_negative_series.quantile(0.75)) / 2, 2
)
embedding_positives_below = (embedding_positive_series < embedding_threshold).sum()
embedding_negatives_above = (embedding_negative_series >= embedding_threshold).sum()
print(f"corte escolhido: {embedding_threshold}")
print(f"pares corretos abaixo do corte: {embedding_positives_below}/{len(embedding_positive_series)}")
print(f"pares aleatórios acima do corte: {embedding_negatives_above}/{len(embedding_negative_series)}")

corte escolhido: 0.47
pares corretos abaixo do corte: 3/35
pares aleatórios acima do corte: 0/35


O corte sai em 0,47, o ponto médio entre o primeiro quartil dos 35 pares confirmados por citação (0,67) e o terceiro quartil dos 35 pares aleatórios (0,27). Nas versões anteriores deste notebook o mesmo método deu 0,46 (31 pares) e 0,44 (33 pares): o conjunto de pares confirmados depende da política de casamento de citação, que mudou na última seção, e os pares aleatórios são re-sorteados por completo a cada versão, porque a semente é fixada uma vez antes das duas calibrações. Com cerca de 35 pares de cada lado, o corte oscila na ordem de 0,02; isso é uma limitação da calibração a registrar junto com o valor. Três pares corretos ficam abaixo do corte e nenhum par aleatório fica acima dele (o maior chega a 0,39).

## Cobertura comparada: citação, TF-IDF e embeddings

In [26]:
def classify_embedding_evidence(opinion_text, person_speech, sentences):
    if find_opinion_quote_evidence(opinion_text, person_speech) is not None:
        return "quote_found"

    best_sentence, score = best_embedding_match(opinion_text, sentences, embedding_model)
    if best_sentence is None:
        return "no_evidence"
    return "semantic_match_high" if score >= embedding_threshold else "semantic_match_weak"

In [27]:
coverage_embeddings = {
    "quote_found": 0,
    "semantic_match_high": 0,
    "semantic_match_weak": 0,
    "person_not_resolved": 0,
}

for hearing in sample_hearings:
    turns = split_into_turns(hearing["transcricao"])
    for person in hearing["metadados"]["envolvidos"]:
        matched_turns, speech = resolve_person_speech(person, turns)
        sentences = split_sentences(speech)
        for opinion_text in person["opinioes"]:
            if not matched_turns:
                coverage_embeddings["person_not_resolved"] += 1
                continue
            coverage_embeddings[classify_embedding_evidence(opinion_text, speech, sentences)] += 1

In [28]:
pd.DataFrame(
    {
        "so_citacao": pd.Series(quote_only_counts),
        "tfidf": pd.Series(coverage_v1),
        "embeddings": pd.Series(coverage_embeddings),
    }
).fillna(0)

,so_citacao,tfidf,embeddings
no_quote,152.0,0.0,0.0
person_not_resolved,13.0,13.0,13.0
quote_found,38.0,38.0,38.0
quote_not_found,65.0,0.0,0.0
semantic_match_high,0.0,153.0,211.0
semantic_match_weak,0.0,64.0,6.0


Com embeddings e o corte de 0,47, das mesmas 268 opiniões: 38 por citação, 211 por similaridade acima do corte, 6 abaixo do corte e 13 de pessoa não resolvida. A camada de baixa confiança cai de 64 opiniões (TF-IDF) para 6, e a de alta confiança sobe de 153 para 211.

## Concordância com a validação manual já feita

Para os 40 pares originais (tiers `semantic_match_high` e `semantic_match_weak`) já classificados manualmente em `udv_manual_review.json`, esta seção recalcula qual sentença o modelo de embeddings escolheria para a mesma opinião, e compara com a evidência que o TF-IDF havia escolhido (a que foi de fato julgada).

In [29]:
with open("udv_manual_review.json") as f:
    review_items = json.load(f)

In [30]:
tfidf_items = [item for item in review_items if item["tier"] in ("semantic_match_high", "semantic_match_weak")]

agreement_count = 0
disagreement_count = 0

for item in tfidf_items:
    hearing = next(h for h in sample_hearings if h["id"] == item["hearing_id"])
    turns = split_into_turns(hearing["transcricao"])
    person = next(p for p in hearing["metadados"]["envolvidos"] if p["nome"] == item["person"])
    _, speech = resolve_person_speech(person, turns)
    sentences = split_sentences(speech)
    embedding_sentence, embedding_score = best_embedding_match(item["opinion"], sentences, embedding_model)

    if embedding_sentence == item["evidence"]:
        agreement_count += 1
    else:
        disagreement_count += 1

In [31]:
print(f"embeddings escolhe a mesma sentença que o TF-IDF: {agreement_count}/{len(tfidf_items)}")
print(f"embeddings escolhe uma sentença diferente: {disagreement_count}/{len(tfidf_items)}")

embeddings escolhe a mesma sentença que o TF-IDF: 15/40
embeddings escolhe uma sentença diferente: 25/40


### Julgamento manual dos casos divergentes

Os 25 casos em que embeddings escolhe uma sentença diferente da que TF-IDF havia escolhido foram julgados manualmente e adicionados a `udv_manual_review.json` sob o tier `embedding_diff`, com a evidência e o julgamento originais do TF-IDF preservados nos campos `tfidf_evidence` e `tfidf_judgment` para comparação.

In [32]:
embedding_diff_items = [item for item in review_items if item["tier"] == "embedding_diff"]
embedding_diff_judgment_counts = {
    judgment: sum(1 for item in embedding_diff_items if item["judgment"] == judgment)
    for judgment in ("correta", "parcial", "incorreta")
}

In [33]:
for judgment, count in embedding_diff_judgment_counts.items():
    print(f"{judgment}: {count}/{len(embedding_diff_items)}")

correta: 9/25
parcial: 10/25
incorreta: 6/25


## Defeitos do resolvedor de nomes e do casamento de citação

A rodada completa do runner em lote (`utils/build_udvs.py`) nas 206 audiências expôs cabeçalhos de turno que a amostra de 20 audiências quase não contém (só a forma do nome social aparece nela, uma vez, na audiência 12), e o resolvedor de nomes da pipeline compartilhada tratava mal quatro formas deles. O sintoma era o `tier` `person_not_resolved`, que afirma "essa pessoa não foi encontrada na transcrição", atribuído a pessoas que falaram entre 9 e 28 vezes na sessão. Esta seção mostra cada forma, mede quantas pessoas só são resolvidas pelas regras adicionadas, quantas citações mudam de resultado com as correções do casamento de citação, e o efeito de descartar marcações de palco.

In [34]:
from utils.udv_pipeline import (
    SENTENCE_BOUNDARY_PATTERN,
    STAGE_DIRECTION_PATTERN,
    enclosing_sentence,
    find_opinion_quote_match,
    find_quote_match,
    is_party_info,
    matching_turns,
    names_match,
    resolve_turn_name,
    turn_name_candidates,
)

In [35]:
header_cases = {
    "abreviatura com ponto no parêntese": (60, "Dr. Zacharias Calil"),
    "nome social entre parênteses, sem ponto": (12, "Pagu Rodrigues"),
    "marcador de tradução no parêntese": (89, "Julissa Mantilla Falcón"),
    "um só token na metadata, nome completo no turno": (109, "Gaguim"),
    "um só token na metadata, nome completo no turno (2)": (133, "Bacelar"),
}
for label, (hearing_id, person_name) in header_cases.items():
    hearing = next(h for h in lds_records if h["id"] == hearing_id)
    turns = split_into_turns(hearing["transcricao"])
    person = next(p for p in hearing["metadados"]["envolvidos"] if p["nome"] == person_name)
    matched_turns, _ = resolve_person_speech(person, turns)
    first_turn = matched_turns[0]
    print(f"{label}: {person_name!r} ({person['cargo']})")
    print(f"    cabeçalho: {first_turn['raw_name']}({first_turn['party_info']})")
    print(f"    candidatos de nome: {turn_name_candidates(first_turn)} | turnos resolvidos: {len(matched_turns)}")


abreviatura com ponto no parêntese: 'Dr. Zacharias Calil' (Deputado (União-GO))
    cabeçalho: PRESIDENTE(Dr. Zacharias Calil. UNIÃO - GO)
    candidatos de nome: ['PRESIDENTE', 'Dr. Zacharias Calil'] | turnos resolvidos: 25
nome social entre parênteses, sem ponto: 'Pagu Rodrigues' (Coordenadora-geral de Prevenção de Violência contra as Mulheres do Ministério das Mulheres)
    cabeçalho: PATRÍCIA RODRIGUES DA SILVA(PAGU RODRIGUES)
    candidatos de nome: ['PATRÍCIA RODRIGUES DA SILVA', 'PAGU RODRIGUES'] | turnos resolvidos: 2
marcador de tradução no parêntese: 'Julissa Mantilla Falcón' (Presidente da Comissão Interamericana dos Direitos Humanos)
    cabeçalho: JULISSA MANTILLA FALCÓN(Manifestação em língua estrangeira. Tradução simultânea.)
    candidatos de nome: ['JULISSA MANTILLA FALCÓN', 'Manifestação em língua estrangeira. Tradução simultânea.'] | turnos resolvidos: 3
um só token na metadata, nome completo no turno: 'Gaguim' (Deputado (União-TO))
    cabeçalho: CARLOS HENRIQUE GAG

In [36]:
def matched_candidate_kinds(person, turns):
    kinds = set()
    for turn in turns:
        party_info = turn["party_info"]
        dotted_party = resolve_turn_name(turn) != turn["raw_name"]
        if names_match(person["nome"], turn["raw_name"]):
            kinds.add("nome_do_cabecalho_atras_de_parentese_pontuado" if ". " in party_info else "nome_do_cabecalho")
        if dotted_party and names_match(person["nome"], resolve_turn_name(turn)):
            kinds.add(
                "parentese_pontuado_com_abreviatura"
                if "." in resolve_turn_name(turn)
                else "parentese_pontuado"
            )
        elif not dotted_party and party_info and not is_party_info(party_info):
            if names_match(person["nome"], party_info):
                kinds.add("alias_sem_ponto")
    return kinds


In [37]:
previously_reachable_kinds = {"nome_do_cabecalho", "parentese_pontuado"}
resolution_by_rule = {
    "regras anteriores": 0,
    "só regras novas de candidato": 0,
    "só regra de token único": 0,
    "não resolvida": 0,
}
new_rule_people = []
for hearing in lds_records:
    turns = split_into_turns(hearing["transcricao"])
    for person in hearing["metadados"]["envolvidos"]:
        matched_turns, _ = resolve_person_speech(person, turns)
        if not matched_turns:
            resolution_by_rule["não resolvida"] += 1
        elif not matching_turns(person["nome"], turns):
            resolution_by_rule["só regra de token único"] += 1
            new_rule_people.append((hearing["id"], person["nome"], len(matched_turns), "token único"))
        elif matched_candidate_kinds(person, turns) & previously_reachable_kinds:
            resolution_by_rule["regras anteriores"] += 1
        else:
            resolution_by_rule["só regras novas de candidato"] += 1
            kinds = ", ".join(sorted(matched_candidate_kinds(person, turns)))
            new_rule_people.append((hearing["id"], person["nome"], len(matched_turns), kinds))
pd.Series(resolution_by_rule, name="pessoas_206_audiencias")


regras anteriores               1003
só regras novas de candidato      15
só regra de token único            2
não resolvida                     45
Name: pessoas_206_audiencias, dtype: int64

In [38]:
pd.DataFrame(new_rule_people, columns=["audiencia", "pessoa", "turnos", "regra"])


,audiencia,pessoa,turnos,regra
0,12,Pagu Rodrigues,2,alias_sem_ponto
1,27,Prof. Paulo Fernando,23,parentese_pontuado_com_abreviatura
2,37,Dr. Zacharias Calil,9,parentese_pontuado_com_abreviatura
3,60,Dr. Zacharias Calil,25,parentese_pontuado_com_abreviatura
4,68,Dr. Zacharias Calil,22,parentese_pontuado_com_abreviatura
5,82,Mestre Alcides,2,alias_sem_ponto
6,82,Mãe Beth de Oxum,3,alias_sem_ponto
7,82,Mestre Chico,3,alias_sem_ponto
8,82,Mestre Paulão Kikongo,2,alias_sem_ponto
9,83,Markinhus Souza,3,alias_sem_ponto


In [39]:
def turns_by_previous_rules(person, turns):
    matched = []
    for turn in turns:
        plain_header = ". " not in turn["party_info"] and names_match(person["nome"], turn["raw_name"])
        resolved_name = resolve_turn_name(turn)
        plain_dotted = (
            resolved_name != turn["raw_name"]
            and "." not in resolved_name
            and names_match(person["nome"], resolved_name)
        )
        if plain_header or plain_dotted:
            matched.append(turn)
    return matched


In [40]:
extended_people = []
unresolved_people = []
for hearing in lds_records:
    turns = split_into_turns(hearing["transcricao"])
    for person in hearing["metadados"]["envolvidos"]:
        matched_turns, _ = resolve_person_speech(person, turns)
        if not matched_turns:
            unresolved_people.append((hearing["id"], person["nome"], person["cargo"][:60]))
            continue
        previous = turns_by_previous_rules(person, turns)
        if previous and len(matched_turns) > len(previous):
            extended_people.append((hearing["id"], person["nome"], len(previous), len(matched_turns), len(person["opinioes"])))
pd.DataFrame(extended_people, columns=["audiencia", "pessoa", "turnos_antes", "turnos_agora", "opinioes"])


,audiencia,pessoa,turnos_antes,turnos_agora,opinioes
0,172,Denildo de Moraes Biko Rodrigues,1,3,4
1,195,Krisztian Katona,1,3,3


In [41]:
pd.set_option("display.max_rows", 60)
pd.DataFrame(unresolved_people, columns=["audiencia", "pessoa", "cargo"])


,audiencia,pessoa,cargo
0,5,Aline Costa,Coordenação-Geral de Saúde da Pessoa com Defic...
1,13,Ricardo Rabelo,Desembargador do Tribunal Regional Federal
2,20,Maíra Pankararu,Conselheira da Comissão de Anistia e advogada
3,20,Adriana Ramos,Consultora do Programa de Política e Direito S...
4,20,Roberta Amanajás,"Professora do Instituto Brasileiro de Ensino, ..."
5,20,Joaquim Paulo de Lima Kaxinawá,Doutor em Linguística pela Universidade de Bra...
6,25,Ricardo Demichelli,Subsecretário de Agricultura de Minas Gerais
7,35,Melillo Diniz Nascimento,Representante do Movimento de Combate à Corrup...
8,44,Daniele Lomba,Gerente de Sustentabilidade e Meio Ambiente da...
9,44,Mauro de Almeida,Secretário de Meio Ambiente do Pará


In [42]:
shared_turn_pairs = []
for hearing in lds_records:
    turns = split_into_turns(hearing["transcricao"])
    matched_by_person = [
        {t["turn_index"] for t in resolve_person_speech(person, turns)[0]}
        for person in hearing["metadados"]["envolvidos"]
    ]
    for index, turns_a in enumerate(matched_by_person):
        for turns_b in matched_by_person[index + 1 :]:
            if turns_a & turns_b:
                shared_turn_pairs.append(hearing["id"])
print(f"pares de participantes da mesma audiência que compartilham algum turno: {len(shared_turn_pairs)}")


pares de participantes da mesma audiência que compartilham algum turno: 0


Nas 206 audiências, 1.020 dos 1.065 participantes listados em `envolvidos` são resolvidos a pelo menos um turno. Desses, 1.003 são alcançados pelas regras anteriores (o nome do cabeçalho, ou o nome sem abreviatura dentro de um parêntese pontuado seguido de filiação partidária, como "Sanderson. PL - RS"), o mesmo total que o runner registrou na rodada de 18/09/2026 antes da correção. Os outros 17 só são resolvidos pelas regras adicionadas: 7 pelo parêntese pontuado com abreviatura ("Dr. Zacharias Calil. UNIÃO - GO", que a regra antiga cortava em "Dr"), 7 pelo nome social entre parênteses sem ponto, 1 pelo nome do cabeçalho quando o parêntese é o marcador de tradução simultânea, e 2 pela regra de token único, que aceita um nome de uma só palavra em `envolvidos` ("Gaguim", "Bacelar") apenas quando exatamente um orador da audiência tem essa palavra no nome; nos dois casos, a UF do `cargo` (TO, BA) coincide com a UF da filiação no cabeçalho. Nenhum turno é atribuído a dois participantes da mesma audiência. Dois participantes que já eram resolvidos ganham turnos com as regras novas (Denildo de Moraes Biko Rodrigues, audiência 172, pelo alias "BIKO RODRIGUES"; Krisztian Katona, audiência 195, pelo marcador de tradução), o que muda a evidência de 4 opiniões sem mudar o nível de nenhuma. A lista dos 45 participantes que continuam sem turno mostra instituições listadas como participante (BNDES, Febraban, Karpowership), grafias diferentes entre metadata e transcrição, e pessoas sem turno de fala no texto.


In [43]:
relaxed_boundary_matches = []
for hearing in lds_records:
    turns = split_into_turns(hearing["transcricao"])
    for person in hearing["metadados"]["envolvidos"]:
        matched_turns, speech = resolve_person_speech(person, turns)
        if not matched_turns:
            continue
        for opinion_text in person["opinioes"]:
            match = find_opinion_quote_match(opinion_text, speech)
            if match is None:
                continue
            hit = quote_prefix_pattern(match["prefix"]).search(speech)
            if hit.start() > 0 and speech[hit.start() - 1].isalnum():
                relaxed_boundary_matches.append((hearing["id"], person["nome"], match["prefix"], match["words"], speech[max(0, hit.start() - 30) : hit.end() + 20]))
pd.DataFrame(relaxed_boundary_matches, columns=["audiencia", "pessoa", "prefixo", "palavras", "contexto_na_fala"])

,audiencia,pessoa,prefixo,palavras,contexto_na_fala
0,83,Carlos Ricardo Junior,Housing First,2,"é uma metodologia, baseada noHousing First,co..."
1,130,Felipe Lopes,Nosso sonho,2,"essa visão de mercado, como oNosso sonho: a h..."


In [44]:
quote_issue_counts = {
    "segunda citação encontrada, primeira não": 0,
    "prefixo casando dentro de palavra maior": 0,
    "evidência não é uma única sentença de 4+ palavras": 0,
}
for hearing in lds_records:
    turns = split_into_turns(hearing["transcricao"])
    for person in hearing["metadados"]["envolvidos"]:
        matched_turns, speech = resolve_person_speech(person, turns)
        if not matched_turns:
            continue
        sentences = split_sentences(speech)
        for opinion_text in person["opinioes"]:
            quotes = extract_quotes(opinion_text)
            found_each = [find_quote_match(quote, speech) for quote in quotes]
            if len(quotes) >= 2 and found_each[0] is None and any(found_each[1:]):
                quote_issue_counts["segunda citação encontrada, primeira não"] += 1
            prefixes = [
                " ".join(quote.split()[:length]) for quote in quotes for length in (10, 6, 4, 3)
            ]
            if any(
                len(prefix) > 5 and prefix in speech and quote_prefix_pattern(prefix).search(speech) is None
                for prefix in prefixes
            ):
                quote_issue_counts["prefixo casando dentro de palavra maior"] += 1
            found = find_opinion_quote_evidence(opinion_text, speech)
            if found is not None and enclosing_sentence(found, speech) not in sentences:
                quote_issue_counts["evidência não é uma única sentença de 4+ palavras"] += 1
pd.Series(quote_issue_counts, name="opinioes_206_audiencias")

segunda citação encontrada, primeira não             9
prefixo casando dentro de palavra maior              3
evidência não é uma única sentença de 4+ palavras    4
Name: opinioes_206_audiencias, dtype: int64

No casamento de citação, três correções. Todas as citações entre aspas de uma opinião são tentadas, não só a primeira: em 9 opiniões das 206 audiências a primeira citação não é localizada e uma posterior é. O prefixo precisa casar em fronteira de palavra, o que descarta 3 casos em que "Nós temos o" casava "Nós temos os" ou "negócio" casava "negócios"; a fronteira esquerda é relaxada quando o trecho casado começa com maiúscula, porque a transcrição às vezes cola o artigo à palavra seguinte, e a tabela acima lista os únicos dois casos em que isso decide o casamento ("noHousing First", "oNosso sonho"), ambos com o texto da citação presente na fala e ambos com prefixo de 2 palavras, curto demais para virar citação pela regra da próxima seção. A evidência gravada é a sentença que contém o prefixo, ou o trecho de sentenças que ele atravessa, em vez do próprio prefixo, que antes ficava como texto da evidência quando nenhuma sentença de 4 palavras ou mais o continha. Em 4 opiniões essa evidência não é uma única sentença de 4 palavras ou mais: citações curtas ("É o BNDES?") ou que atravessam uma fronteira de sentença.

In [45]:
stage_direction_people = []
for hearing in lds_records:
    turns = split_into_turns(hearing["transcricao"])
    for person in hearing["metadados"]["envolvidos"]:
        matched_turns, speech = resolve_person_speech(person, turns)
        if not matched_turns:
            continue
        long_parts = [part for part in SENTENCE_BOUNDARY_PATTERN.split(speech) if len(part.split()) >= 4]
        stage_parts = [part for part in long_parts if STAGE_DIRECTION_PATTERN.match(part)]
        if stage_parts:
            stage_direction_people.append((hearing["id"], person["nome"], len(person["opinioes"]), len(split_sentences(speech)), stage_parts[0]))
pd.DataFrame(stage_direction_people, columns=["audiencia", "pessoa", "opinioes", "sentencas_restantes", "marcacao_descartada"])


,audiencia,pessoa,opinioes,sentencas_restantes,marcacao_descartada
0,53,Renata Rezende,2,0,(Manifestação em LIBRAS.) (Manifestação em LIB...
1,111,Rodrigo Rosso Marques,4,0,(Manifestação em LIBRAS.) (Manifestação em LIB...
2,111,Messias Ramos Costa,2,0,(Manifestação em LIBRAS.) (Manifestação em LIB...


Marcações de palco entre parênteses, como "(Manifestação em LIBRAS.)", não são fala. Antes desta correção, `split_sentences` as tratava como sentença quando tinham 4 palavras ou mais, o que acontecia quando dois turnos só com a marcação eram concatenados; para os 3 participantes acima, que só falaram em LIBRAS e não têm texto na transcrição, a marcação virava a evidência de todas as suas 8 opiniões, 3 delas acima do corte de similaridade. Descartar essas partes deixa esses participantes sem sentença, e o runner passa a classificar essas 8 opiniões como `no_evidence`. Nas 206 audiências, as 3 marcações desta tabela são as únicas partes descartadas por essa regra.


## Política de casamento de citação

O casamento de citação era sensível a maiúsculas e minúsculas, e a matéria costuma iniciar a citação com maiúscula onde a transcrição tem a mesma frase no meio de um período ("Confisco de aposentados é uma covardia" contra "O confisco de aposentados é uma covardia"). Ignorar a caixa recupera esses casos, mas expõe um problema que já existia: a busca desce uma escada de prefixos de 10, 6, 4 e 3 palavras, e os degraus curtos são frases de ligação ("Eu acho que", "Do ponto de vista") que casam em quase qualquer fala, apontando uma sentença que não é a origem da citação.

A política adotada separa os dois efeitos: o prefixo passa a casar ignorando a caixa, e só um prefixo de 6 palavras ou mais produz o nível `quote_found`. Um prefixo curto que casa não é jogado fora: quando ele cai na mesma sentença que o encoder escolheu para aquela opinião (ou numa que a contenha), a evidência continua sendo a sentença do encoder, o nível continua saindo do score, e a evidência guarda os dois sinais juntos, com `support_type` `semantic_with_short_quote`, o prefixo e o score. As células abaixo medem, nas 206 audiências, quantas opiniões cada parte da regra move, comparando o casamento sensível à caixa (`utils.measure_case_insensitive_quotes`, modo `exact`, que reproduz o comportamento anterior) com o casamento da pipeline atual.

In [46]:
from collections import Counter

from utils.measure_case_insensitive_quotes import case_sensitive_prefix_pattern, find_opinion_match
from utils.udv_pipeline import TRUSTED_PREFIX_WORDS, is_trusted_quote

In [47]:
quote_matches = []
for hearing in lds_records:
    turns = split_into_turns(hearing["transcricao"])
    for person in hearing["metadados"]["envolvidos"]:
        matched_turns, speech = resolve_person_speech(person, turns)
        if not matched_turns:
            continue
        for opinion_text in person["opinioes"]:
            if not extract_quotes(opinion_text):
                continue
            exact_match = find_opinion_match(opinion_text, speech, "exact")
            relaxed_match = find_opinion_quote_match(opinion_text, speech)
            quote_matches.append(
                {
                    "audiencia": hearing["id"],
                    "pessoa": person["nome"],
                    "palavras_exato": exact_match["prefix_words"] if exact_match else 0,
                    "palavras_ignorando_caixa": relaxed_match["words"] if relaxed_match else 0,
                }
            )
quote_matches_df = pd.DataFrame(quote_matches)
print(f"opiniões com citação entre aspas, de pessoas resolvidas: {len(quote_matches_df)}")

opiniões com citação entre aspas, de pessoas resolvidas: 899


In [48]:
pd.crosstab(
    quote_matches_df["palavras_exato"].rename("palavras no prefixo, casamento exato"),
    quote_matches_df["palavras_ignorando_caixa"].rename("palavras no prefixo, ignorando a caixa"),
)

"palavras no prefixo, ignorando a caixa",0,1,2,3,4,5,6,7,10
"palavras no prefixo, casamento exato",,,,,,,,,
0,395,0,0,50,78,0,61,2,54
1,0,2,0,0,0,0,0,0,0
2,0,0,10,0,0,0,0,0,0
3,0,0,0,40,5,0,5,0,5
4,0,0,0,0,58,0,2,0,0
5,0,0,0,0,0,1,0,0,0
6,0,0,0,0,0,0,77,0,3
7,0,0,0,0,0,0,0,1,0
10,0,0,0,0,0,0,0,0,50


In [49]:
accepted_before = quote_matches_df["palavras_exato"] > 0
accepted_now = quote_matches_df["palavras_ignorando_caixa"] >= TRUSTED_PREFIX_WORDS
pd.crosstab(
    accepted_before.rename("aceita pela regra anterior"),
    accepted_now.rename("aceita pela regra nova"),
)

aceita pela regra nova,False,True
aceita pela regra anterior,,
False,523,117
True,116,143


In [50]:
rescued = quote_matches_df[
    accepted_before & accepted_now & (quote_matches_df["palavras_exato"] < TRUSTED_PREFIX_WORDS)
]
print(f"casamentos exatos de prefixo curto mantidos por casarem um prefixo mais longo: {len(rescued)}")
rescued.groupby(["palavras_exato", "palavras_ignorando_caixa"]).size()

casamentos exatos de prefixo curto mantidos por casarem um prefixo mais longo: 12


palavras_exato  palavras_ignorando_caixa
3               6                           5
                10                          5
4               6                           2
dtype: int64

In [51]:
first_quote_limits = []
for hearing in lds_records:
    turns = split_into_turns(hearing["transcricao"])
    for person in hearing["metadados"]["envolvidos"]:
        matched_turns, speech = resolve_person_speech(person, turns)
        if not matched_turns:
            continue
        for opinion_text in person["opinioes"]:
            match = find_opinion_quote_match(opinion_text, speech)
            if match is None or is_trusted_quote(match):
                continue
            later = [
                found
                for found in (find_quote_match(quote, speech) for quote in extract_quotes(opinion_text))
                if found is not None and is_trusted_quote(found)
            ]
            if later:
                first_quote_limits.append((hearing["id"], person["nome"], match["prefix"], later[0]["prefix"]))
pd.DataFrame(first_quote_limits, columns=["audiencia", "pessoa", "prefixo_da_primeira_citacao", "prefixo_de_citacao_posterior"])

,audiencia,pessoa,prefixo_da_primeira_citacao,prefixo_de_citacao_posterior
0,1,Eli Vieira Junior,Esses termos são,"É a lei que deve mandar, e não a opinião"
1,6,Ramiro Madureira,Acreditávamos que o custo,"Ao contrário do que prevíamos, o mercado tem s..."
2,183,Rodrigo Agostinho,nenhuma perseguição,"Nenhuma empresa brasileira, nenhum empreendedo..."


In [52]:
with open("artifacts/udv/udv_v0_coverage.json") as f:
    udv_v0_coverage = json.load(f)
pd.Series(udv_v0_coverage["opinions"]["by_tier"], name="opinioes_por_tier")

quote_found             260
semantic_match_high    1793
semantic_match_weak      52
no_evidence               8
person_not_resolved      90
Name: opinioes_por_tier, dtype: int64

In [53]:
pd.Series(udv_v0_coverage["evidence_support_types"], name="evidencias_por_tipo_de_suporte")

direct_quote                  260
semantic_with_short_quote     100
semantic_similarity          1745
Name: evidencias_por_tipo_de_suporte, dtype: int64

In [54]:
with open("artifacts/udv/udv_v0.jsonl") as f:
    udv_v0_records = [json.loads(line) for line in f]
short_quote_df = pd.DataFrame(
    [
        {
            "tier": record["tier"],
            "palavras_no_prefixo": len(record["evidence"]["quote_prefix"].split()),
            "score": record["evidence"]["score"],
        }
        for record in udv_v0_records
        if record["evidence"] and record["evidence"]["support_type"] == "semantic_with_short_quote"
    ]
)
short_quote_df.groupby(["tier", "palavras_no_prefixo"]).agg(
    registros=("score", "size"), score_minimo=("score", "min"), score_mediano=("score", "median")
).round(3)

registros  score_minimo  \
tier                palavras_no_prefixo                            
semantic_match_high 2                            4         0.494   
                    3                           34         0.565   
                    4                           61         0.561   
                    5                            1         0.686   

                                         score_mediano  
tier                palavras_no_prefixo                 
semantic_match_high 2                            0.570  
                    3                            0.790  
                    4                            0.750  
                    5                            0.686

In [55]:
speech_by_person = {}
for hearing in lds_records:
    turns = split_into_turns(hearing["transcricao"])
    for person_index, person in enumerate(hearing["metadados"]["envolvidos"]):
        matched_turns, speech = resolve_person_speech(person, turns)
        if matched_turns:
            speech_by_person[(hearing["id"], person_index)] = speech
print(f"falas resolvidas nas 206 audiências: {len(speech_by_person)}")

falas resolvidas nas 206 audiências: 1020


In [56]:
resolved_speeches = list(speech_by_person.values())
short_quote_reach = []
for record in udv_v0_records:
    evidence = record["evidence"]
    if not evidence or evidence["support_type"] != "semantic_with_short_quote":
        continue
    pattern = quote_prefix_pattern(evidence["quote_prefix"])
    speeches_with_prefix = sum(1 for speech in resolved_speeches if pattern.search(speech))
    short_quote_reach.append((record["id"], evidence["quote_prefix"], speeches_with_prefix))
short_quote_reach_df = pd.DataFrame(short_quote_reach, columns=["udv", "prefixo", "falas_com_o_prefixo"])
print(f"prefixos que só aparecem na fala do próprio participante: {(short_quote_reach_df['falas_com_o_prefixo'] == 1).sum()}/{len(short_quote_reach_df)}")
print(f"prefixos presentes em 10 ou mais das {len(resolved_speeches)} falas: {(short_quote_reach_df['falas_com_o_prefixo'] >= 10).sum()}")
short_quote_reach_df.nlargest(5, "falas_com_o_prefixo")

prefixos que só aparecem na fala do próprio participante: 63/100
prefixos presentes em 10 ou mais das 1020 falas: 13


,udv,prefixo,falas_com_o_prefixo
58,udv-106-2-0,Eu acho que,417
70,udv-139-0-0,Nós estamos falando,136
83,udv-173-0-0,É um grande,87
30,udv-61-4-0,Hoje nós temos,68
12,udv-24-0-0,Em vez de,66


In [57]:
def person_key(record):
    _, hearing_id, person_index, _ = record["id"].split("-")
    return int(hearing_id), int(person_index)


def opinion_tokens(text):
    return {token for token in (re.sub(r"\W", "", word.lower()) for word in text.split()) if token}


repeated_prefix = []
for record in udv_v0_records:
    evidence = record["evidence"]
    if not evidence or not evidence["quote_prefix"]:
        continue
    speech = speech_by_person[person_key(record)]
    prefix = evidence["quote_prefix"]
    hits = list(quote_prefix_pattern(prefix).finditer(speech))
    if len(hits) < 2:
        continue
    wanted = opinion_tokens(record["proposition"])
    overlaps = [
        len(wanted & opinion_tokens(speech[hit.start() : hit.start() + 400])) for hit in hits
    ]
    repeated_prefix.append(
        {
            "udv": record["id"],
            "support_type": evidence["support_type"],
            "ocorrencias": len(hits),
            "sobreposicao_da_gravada": overlaps[0],
            "melhor_sobreposicao": max(overlaps),
            "tambem_repetido_no_modo_exato": len(case_sensitive_prefix_pattern(prefix).findall(speech)) > 1,
        }
    )
repeated_prefix_df = pd.DataFrame(repeated_prefix)
print(f"prefixos gravados que ocorrem mais de uma vez na fala: {len(repeated_prefix_df)}")
print(f"desses, já repetidos no casamento sensível à caixa: {repeated_prefix_df['tambem_repetido_no_modo_exato'].sum()}")
print(f"registros em que outra ocorrência tem sobreposição maior com a opinião: {(repeated_prefix_df['sobreposicao_da_gravada'] < repeated_prefix_df['melhor_sobreposicao']).sum()}")
repeated_prefix_df[repeated_prefix_df["sobreposicao_da_gravada"] < repeated_prefix_df["melhor_sobreposicao"]]

prefixos gravados que ocorrem mais de uma vez na fala: 13
desses, já repetidos no casamento sensível à caixa: 4
registros em que outra ocorrência tem sobreposição maior com a opinião: 1


,udv,support_type,ocorrencias,sobreposicao_da_gravada,melhor_sobreposicao,tambem_repetido_no_modo_exato
3,udv-65-1-0,direct_quote,2,12,15,True


In [58]:
udv_v0_by_id = {record["id"]: record for record in udv_v0_records}
direct_test_only = []
for (hearing_id, person_index), speech in speech_by_person.items():
    hearing = next(h for h in lds_records if h["id"] == hearing_id)
    person = hearing["metadados"]["envolvidos"][person_index]
    for opinion_index, opinion_text in enumerate(person["opinioes"]):
        record = udv_v0_by_id[f"udv-{hearing_id}-{person_index}-{opinion_index}"]
        evidence = record["evidence"]
        if not evidence or evidence["support_type"] != "semantic_similarity":
            continue
        match = find_opinion_quote_match(opinion_text, speech)
        if match is None or is_trusted_quote(match):
            continue
        if quote_prefix_pattern(match["prefix"]).search(evidence["text"]):
            direct_test_only.append((record["id"], match["prefix"]))
print(f"registros que ganhariam corroboração se o teste fosse o prefixo dentro da sentença gravada: {len(direct_test_only)}")
pd.DataFrame(direct_test_only, columns=["udv", "prefixo"])

registros que ganhariam corroboração se o teste fosse o prefixo dentro da sentença gravada: 7


,udv,prefixo
0,udv-34-0-2,É por isso que
1,udv-51-3-0,A curricularização da extensão
2,udv-83-3-1,Na cidade de São
3,udv-100-2-1,Eu acho interessante que
4,udv-115-0-0,Esses gases se acumulam
5,udv-130-3-0,Nosso sonho
6,udv-200-0-0,Do ponto de vista


In [59]:
quote_prefix_words = Counter(
    len(record["evidence"]["quote_prefix"].split())
    for record in udv_v0_records
    if record["tier"] == "quote_found"
)
pd.Series(dict(sorted(quote_prefix_words.items())), name="quote_found_por_palavras_no_prefixo")

6     145
7       3
10    112
Name: quote_found_por_palavras_no_prefixo, dtype: int64

Das 899 opiniões com citação entre aspas de pessoas resolvidas, 395 não casam em nenhum dos dois modos. Ignorando a caixa, 245 passam a casar (50 com prefixo de 3 palavras, 78 de 4, 61 de 6, 2 de 7 e 54 de 10) e nenhum casamento sensível à caixa se perde. Pela regra das 6 palavras, dessas 245 só 117 viram `quote_found`.

O efeito conjunto, na tabela 2x2: 143 opiniões eram citação e continuam sendo, 116 deixam de ser (todas de prefixo curto), 117 passam a ser, e 523 continuam sem citação aceita. As 12 que ficam em pé por outro caminho são casamentos que eram curtos no modo exato e, ao ignorar a caixa, casam um prefixo mais longo (3 para 6 palavras em 5 casos, 3 para 10 em 5, 4 para 6 em 2). O total fica em 260 `quote_found`, todos com prefixo de 6 palavras ou mais: 145 de 6, 3 de 7 e 112 de 10.

O artefato completo (`artifacts/udv/udv_v0.jsonl`, gerado pelo runner com o corte de 0,47 calibrado acima) fica com 260 `quote_found`, 1.793 `semantic_match_high`, 52 `semantic_match_weak`, 8 `no_evidence` e 90 `person_not_resolved`; por tipo de suporte, 260 `direct_quote`, 100 `semantic_with_short_quote` e 1.745 `semantic_similarity`. Os 100 registros com prefixo curto corroborando a sentença do encoder estão todos acima do corte (score mínimo 0,494), o que era esperado: são casos em que dois sinais independentes apontam a mesma sentença.

Três limitações ficam registradas. A primeira citação da opinião que casa é a que decide, mesmo quando uma citação posterior casaria com prefixo mais longo: são as 3 opiniões da tabela acima, duas terminando com corroboração de prefixo curto e uma (audiência 183) só com similaridade, porque o prefixo curto caiu em outra sentença. O prefixo curto que corrobora não precisa ser distintivo: 99 dos 100 têm 4 palavras ou menos e, embora 63 deles só apareçam na fala do próprio participante, 13 aparecem em 10 ou mais das 1.020 falas resolvidas, com "Eu acho que" em 417 delas. Para esses, a coincidência entre o prefixo e a sentença do encoder diz pouco, e é por isso que o registro fica no nível dado pelo score e não em `quote_found`. Some-se a isso que a evidência é sempre a primeira ocorrência do prefixo na fala: ignorar a caixa faz 13 prefixos gravados ocorrerem mais de uma vez, contra 4 no casamento sensível à caixa, e em 1 deles outra ocorrência tem sobreposição maior com a opinião, um `direct_quote` cujos offsets apontam a passagem errada. O critério de corroboração também depende dessa primeira ocorrência: testar diretamente se o prefixo cai dentro da sentença gravada acrescentaria 7 registros, decisão que fica em aberto porque a projeção de 20/09 que sustenta a política foi calculada com o critério atual. E a precisão dos casamentos aceitos por essa regra ainda não foi checada por leitura humana: o material para isso é `artifacts/udv/case_insensitive_review_template.json`, uma amostra de 40 casamentos com o campo de julgamento vazio, sorteada antes desta regra, em que 21 pares caem no estrato aceito como `quote_found`, 5 viraram corroboração de prefixo curto e 14, similaridade pura.

### Efeito na amostra de validação manual

In [60]:
stale_review_items = []
newly_quoted_items = []
for item in review_items:
    hearing = next(h for h in sample_hearings if h["id"] == item["hearing_id"])
    turns = split_into_turns(hearing["transcricao"])
    person = next(p for p in hearing["metadados"]["envolvidos"] if p["nome"] == item["person"])
    _, speech = resolve_person_speech(person, turns)
    sentences = split_sentences(speech)
    if item["tier"] == "embedding_diff":
        current_evidence, _ = best_embedding_match(item["opinion"], sentences, embedding_model)
    else:
        current_evidence, _ = best_semantic_match(item["opinion"], sentences)
    if current_evidence != item["evidence"]:
        stale_review_items.append((item["tier"], item["person"]))
    if find_opinion_quote_evidence(item["opinion"], speech) is not None:
        newly_quoted_items.append((item["tier"], item["person"], item["judgment"]))
print(f"pares da amostra manual cuja evidência de similaridade mudou: {len(stale_review_items)}/{len(review_items)}")
print(f"pares que a regra de citação nova classificaria como quote_found: {len(newly_quoted_items)}/{len(review_items)}")
pd.DataFrame(newly_quoted_items, columns=["tier", "pessoa", "julgamento"])

pares da amostra manual cuja evidência de similaridade mudou: 0/65
pares que a regra de citação nova classificaria como quote_found: 4/65


,tier,pessoa,julgamento
0,semantic_match_high,Alfredo Gaspar,correta
1,semantic_match_high,Otávio Damaso,correta
2,semantic_match_high,José Almir Cirilo,correta
3,embedding_diff,José Almir Cirilo,correta


Os 65 pares de `udv_manual_review.json` foram sorteados antes destas correções, com o resolvedor e o casamento de citação anteriores. Regenerar a amostra agora (`utils/generate_udv_manual_review.py`) sortearia pares diferentes, porque os pools mudaram. A célula acima verifica o que isso faz com os julgamentos: a evidência de similaridade que a pipeline escolhe hoje é a mesma que foi julgada nos 65 pares, então nenhum julgamento fica inválido, e 4 deles (3 de `semantic_match_high`, 1 de `embedding_diff`, os quatro julgados como corretos) passam a ter citação localizada pela regra nova e deixam de ser produzidos pela camada de similaridade. O arquivo fica congelado como conjunto de validação desta amostra, e a amostra de `semantic_match_high` que ele documenta descreve 17 dos 20 pares originais depois da mudança.

## Conclusão

Dos 25 casos em que embeddings escolhe uma sentença diferente da que TF-IDF havia escolhido, o julgamento manual classificou 9 como corretas, 10 como parciais e 6 como incorretas: a escolha alternativa de embeddings acerta ou confirma parcialmente a opinião em 19 dos 25 casos (76%) e erra completamente em 6 (24%). Em alguns casos (o pedido de desculpas citado por Ramiro Madureira, a falta de contraditório apontada por Arlindo Chinaglia, a posição de Capitão Alberto Neto sobre o saque-aniversário), a sentença escolhida por embeddings é mais específica e mais alinhada ao conteúdo da opinião do que a sentença que TF-IDF tinha escolhido. Em outros (o relatório do apagão citado por Luiz Carlos Ciocchi, os compradores de passagens em 2024 citados por Ramiro Madureira), a sentença escolhida trata de um tema relacionado, mas não sustenta a afirmação específica da opinião.

Somando aos 15 casos em que embeddings concorda com TF-IDF, os 65 pares de `udv_manual_review.json` cobrem todos os casos de divergência entre os dois métodos nesta amostra de validação manual. A pipeline de citação e embeddings roda nas 206 audiências pelo runner em lote `utils/build_udvs.py`, agora com a política de casamento de citação desta última seção e com o corte de 0,47 calibrado aqui, e a cobertura da rodada está em `artifacts/udv/udv_v0_coverage.json`. Duas coisas seguem em aberto: medir concordância fora das 20 audiências desta amostra, e julgar à mão os 40 casamentos de `artifacts/udv/case_insensitive_review_template.json` para medir a precisão da regra de citação adotada.